# Country aid prioritization — an industry-style walkthrough

This is *your* clustering project, rebuilt as a clean, fully-annotated reference. Every decision is explained, every question you raised is answered, and where there's an "industry way" of doing something, it's marked.

**The business problem (framed the way a company would frame it)**
> A humanitarian NGO has a fixed budget. It wants to direct aid to the countries in the *worst* socio-economic and health situation. We have data on 167 countries (child mortality, income, life expectancy, GDP, etc.). Group the countries by overall development level, identify the group most in need, and produce a ranked priority list.

This is an **unsupervised** problem: there's no "correct answer" column to predict. We're *discovering structure*, not predicting a target.

---
### How to read this notebook
- Markdown cells = the *why* (the thinking). Read these slowly.
- Code cells = the *how*. Comments inside explain each line.
- Cells marked **❓ Your question** answer something you specifically asked.
- Cells marked **🏭 Industry note** explain what changes in a real company setting.


## §0 The map — every ML project has this shape

Before any code, hold this in your head. The whole field fits in one pipeline:

| Stage | This project | What you'll do |
|---|---|---|
| 1. Frame the problem | "Find countries most in need" | Decide what success means |
| 2. Get & inspect data | Load the CSV, check types/nulls | `read_csv`, `info`, `describe` |
| 3. Understand the data (EDA) | Histograms, skew, correlations | This is where distributions matter |
| 4. Preprocess / engineer | Log-transform skewed cols, scale | Make data model-ready |
| 5. Model | PCA (optional) + K-Means | Discover the groups |
| 6. Evaluate | Elbow, silhouette, others | Decide how many clusters |
| 7. Interpret & deliver | Cluster profiles, ranked list, map | Turn output into a decision |
| (8. Deploy & monitor) | *not needed here* | Production-only stage |

Notice: **modeling (step 5) is a small slice.** Most real work is steps 2–4 and 7. That's true in industry too.


## §1 Setup

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualization (interactive plots)
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Statistics
from scipy.stats import skew

# Machine learning (scikit-learn)
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
)

# Keep results reproducible — same random seed = same answer every run.
# 🏭 Industry note: ALWAYS set random_state. A model that gives different
# results each run is impossible to debug, review, or trust.
RANDOM_STATE = 42

print("Libraries loaded.")


## §2 Load and inspect the data

First contact with any dataset is always the same three questions:
1. What shape is it? (rows × columns)
2. What are the column types? (numbers vs text)
3. Is it clean? (missing values, duplicates)


In [ ]:
# Load the CSV (make sure Country-data.csv is in the same folder as this notebook)
df = pd.read_csv('Country-data.csv')

# Question 1: shape
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()


In [ ]:
# Question 2: types and non-null counts in one view
df.info()


In [ ]:
# Statistical summary — get a feel for the scale of each column.
# Look at 'mean' vs 'max': if max is WAY bigger than mean, the column is
# right-skewed (a few huge values pulling the tail). Watch 'income' and 'gdpp'.
df.describe()


In [ ]:
# Question 3: is it clean?
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# This dataset is already clean (no nulls, no dupes). Lucky.
# 🏭 Industry note: real data is NEVER this clean. Expect to spend
# most of a project here — fixing nulls, bad encodings, duplicates,
# inconsistent units. Clean data is the exception, not the rule.


## §3 Understand the data — and where distributions come in

This is the stage where everything you learned about distributions pays off. We plot each column's shape and *read* it.


In [ ]:
# Separate the text column (country names) from the numbers.
# We can only do math on numbers.
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric columns we'll analyze:", numerical_cols)


In [ ]:
# Plot every numeric column's distribution in a 3x3 grid.
# This single view tells you the 'shape' of each feature.
fig = make_subplots(rows=3, cols=3, subplot_titles=numerical_cols)
for i, col in enumerate(numerical_cols):
    row, c = i // 3 + 1, i % 3 + 1
    fig.add_trace(go.Histogram(x=df[col], nbinsx=30), row=row, col=c)
fig.update_layout(height=750, showlegend=False, title_text="Feature distributions")
fig.show()


**Reading these shapes (the music):**
- `income`, `gdpp`, `inflation` — bunched at the low end with a long tail stretching right. That's the **log-normal / right-skewed** fingerprint. Classic "rich get richer" multiplicative quantities.
- `child_mort`, `total_fer` — also right-skewed.
- `life_expec` — skewed the *other* way (a tail to the *left*; most countries are high, a few low).
- `health` — roughly symmetric, closest to a bell.

Why do we care? Because two of our tools — **PCA and K-Means both rely on distances**, and distance gets distorted by skew and by features on wildly different scales. So we'll fix skew (log-transform) and scale (standardize) before modeling. Next section.


### ❓ Your question: "Why log-transform only *some* columns? By eye, or by number? Can a computer choose?"

This was the best question in your original notebook. Here's the full answer — **three ways** to decide.


In [ ]:
# WAY 1 — by number. There's a statistic called 'skewness' that measures
# how lopsided a distribution is:
#   skew  ~  0    -> symmetric (bell-like)
#   skew  >  0    -> right tail  (log-transform candidate)
#   skew  <  0    -> left tail
# Rule of thumb: |skew| > 1 is strongly skewed, > 0.5 is moderate.

skewness = df[numerical_cols].apply(skew).sort_values(ascending=False)
print("Skewness of each column (most skewed at top):")
print(skewness.round(2))


So instead of guessing by eye, we let the **number** pick. Any column with `skew > 1` is a strong candidate for log-transforming. That typically flags `income`, `gdpp`, `inflation`, `total_fer`, `child_mort` for this dataset — confirming your eyeball instinct, but now with a rule you can defend.


In [ ]:
# WAY 2 — pick features programmatically using a skew threshold.
# No more hand-picking. The threshold is your one decision.
SKEW_THRESHOLD = 1.0
skewed_features = skewness[skewness > SKEW_THRESHOLD].index.tolist()
print(f"Columns with skew > {SKEW_THRESHOLD} (will log-transform):")
print(skewed_features)


**WAY 3 — let a computer choose *and* apply the transform automatically.**

`PowerTransformer` (Yeo-Johnson) inspects each column and applies the best power transform to make it as bell-shaped as possible — no manual feature list at all.

🏭 Industry note: this is increasingly the preferred approach. It's automatic, it handles negative values (plain `log` can't — and your `inflation` column *could* contain negatives), and it's reproducible. We'll use the manual log-transform below for transparency (so you can see what's happening), then show the `PowerTransformer` one-liner as the professional alternative.


## §4 Preprocess — make the data model-ready

Two steps: (a) reduce skew with a log-transform, (b) put all features on the same scale.


In [ ]:
# Set country names aside; model only on the numbers.
country_names = df['country']
X = df.drop('country', axis=1).copy()

# (a) Log-transform the skewed columns chosen by the rule above.
# np.log1p(x) computes log(1 + x). The '+1' safely handles zeros
# (log(0) is undefined; log1p(0) = 0).
for col in skewed_features:
    X[col] = np.log1p(X[col])

print("Skewness AFTER log-transform (should be closer to 0):")
print(X.apply(skew).round(2))


See how the skewed columns dropped toward 0? The long tails got pulled in. The data now looks more bell-like, which distance-based methods prefer.


In [ ]:
# (b) Standardize: rescale every column to mean=0, std=1.
# WHY THIS MATTERS: K-Means and PCA measure distance. Without scaling,
# 'income' (range ~0-125000) would completely dominate 'health' (range ~1-18)
# simply because its numbers are bigger -- not because it's more important.
# Standardizing puts every feature on equal footing.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("After standardizing (mean~0, std~1 for every column):")
print(X_scaled.describe().round(2).loc[['mean', 'std']])


### 🏭 Industry note: the professional way to chain these steps — `Pipeline`

In your notebook (and above), preprocessing is a series of manual steps. In industry, you'd wrap them in a scikit-learn `Pipeline`. Why? Because a Pipeline (1) guarantees the exact same steps run on new data, (2) prevents subtle bugs like scaling the test set using the training set's statistics, and (3) is a single object you can save and deploy.

You don't *need* it for this exploratory notebook, but here's what it looks like so you recognize it later:


In [ ]:
from sklearn.pipeline import Pipeline

# The same preprocessing as above, but as ONE reusable object.
# PowerTransformer replaces the manual log-transform AND the scaler at once
# (it transforms toward normality and standardizes).
preprocessing = Pipeline(steps=[
    ('power', PowerTransformer(method='yeo-johnson', standardize=True)),
])

# .fit_transform learns the parameters and applies them in one call.
X_prepared = preprocessing.fit_transform(X.copy())
print("Pipeline output shape:", X_prepared.shape)
print("This X_prepared is equivalent (and cleaner) than the manual steps above.")

# For the rest of the notebook we'll continue with X_scaled (the manual version)
# so the steps stay visible and easy to follow.


## §5 ❓ Your question: "Is PCA even necessary?"

**Short answer: no, not for this dataset.** Let me prove it rather than just assert it.

**What PCA does** (you already understand this): it finds new axes — directions in the data where variance (information) is greatest. The first principal component points along the direction of most spread, the second along the next-most, and so on. Higher eigenvalue = more information in that direction. You can then keep just the top few axes and drop the rest, reducing many features down to a handful.

**When PCA actually helps:**
- You have *many* features (hundreds/thousands) — speed and the "curse of dimensionality."
- Features are highly redundant/correlated — PCA compresses them.
- You need to *visualize* high-dimensional data in 2D or 3D.

**Why it's optional here:** you have only 9 features and 167 rows. K-Means handles that directly with no trouble. The main thing PCA buys us is the ability to *draw* the clusters in 2D/3D.

⚠️ A subtlety worth knowing: PCA maximizes *variance*, not *cluster separation*. Occasionally the direction that best separates groups is a low-variance one that PCA throws away — so PCA before clustering can even *hurt*. That's why "always PCA first" is a myth. Let's test whether it matters here by clustering **both ways**.


In [ ]:
# First, let's look at the PCA variance breakdown (the scree analysis).
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)

explained = pca_full.explained_variance_ratio_     # % info per component
cumulative = np.cumsum(explained)                   # running total

print("Information (variance) captured by each principal component:")
for i, (e, c) in enumerate(zip(explained, cumulative), 1):
    print(f"  PC{i}: {e*100:5.1f}%   (cumulative: {c*100:5.1f}%)")


In [ ]:
# Scree plot: bars = info per component, line = cumulative.
# The dashed line marks 95% -- a common 'keep enough components to
# explain 95% of the information' cutoff.
fig = go.Figure()
fig.add_bar(x=[f'PC{i+1}' for i in range(len(explained))], y=explained,
            name='Per component', marker_color='steelblue')
fig.add_scatter(x=[f'PC{i+1}' for i in range(len(cumulative))], y=cumulative,
                name='Cumulative', mode='lines+markers',
                line=dict(color='crimson', width=3), yaxis='y2')
fig.add_hline(y=0.95, line_dash='dash', line_color='gray',
              annotation_text='95%', yref='y2')
fig.update_layout(
    title='PCA scree plot — how many components to keep?',
    xaxis_title='Principal component',
    yaxis=dict(title='Variance ratio'),
    yaxis2=dict(title='Cumulative', overlaying='y', side='right', range=[0, 1.05]),
    height=450)
fig.show()


Read the red line: find where it crosses 95%. For this data that's around **5 components**. So PCA(5) keeps ~95% of the information while dropping from 9 dimensions to 5.

Now the real test — **does clustering actually change if we skip PCA?**


In [ ]:
# Cluster the FULL scaled data (no PCA), K=4
km_no_pca = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=RANDOM_STATE)
labels_no_pca = km_no_pca.fit_predict(X_scaled)

# Cluster the PCA-reduced data (5 components), K=4
pca5 = PCA(n_components=5, random_state=RANDOM_STATE)
X_pca5 = pca5.fit_transform(X_scaled)
km_pca = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=RANDOM_STATE)
labels_pca = km_pca.fit_predict(X_pca5)

# How much do the two clusterings agree? adjusted_rand_score:
#   1.0 = identical groupings, 0 = random agreement.
agreement = adjusted_rand_score(labels_no_pca, labels_pca)
print(f"Agreement between 'no PCA' and 'with PCA' clusterings: {agreement:.3f}")
print("(Close to 1.0 means PCA made almost no difference to the grouping.)")

# A crosstab shows it concretely: each row should map cleanly to one column.
print("\nHow the two clusterings line up:")
print(pd.crosstab(labels_no_pca, labels_pca,
                  rownames=['No PCA'], colnames=['With PCA']))


If the agreement score is high (it will be), you've **proven** PCA wasn't doing the real work here — the groups are essentially the same with or without it. PCA's only real contribution for us is the 2D/3D picture. That's a legitimate reason to keep it (humans need pictures), but now you *know* it's a visualization choice, not a modeling necessity.

For the rest of the notebook we'll use the PCA version, purely so we can plot the clusters in 2-D and 3-D.


## §6 ❓ Your question: "Are elbow & silhouette the best? Am I using outdated methods?"

Here's the honest truth: **clustering has no ground truth.** Nobody labeled these countries with their "correct" cluster, so there is no single objectively-best number of clusters. Every method below is a *heuristic* — a reasonable guess, not a proof. Good practitioners look at several and combine them with business sense.

Let me show you **four** metrics, not two, so you see the full toolkit:


In [ ]:
K_range = range(2, 11)

# We'll collect four different 'how good is this K?' metrics.
wcss = []        # Elbow: within-cluster sum of squares (lower, look for the 'elbow')
silhouette = []  # Silhouette: -1 to 1, higher = better separated
davies = []      # Davies-Bouldin: lower = better
calinski = []    # Calinski-Harabasz: higher = better

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_pca5)
    wcss.append(km.inertia_)                              # inertia_ IS the WCSS
    silhouette.append(silhouette_score(X_pca5, labels))
    davies.append(davies_bouldin_score(X_pca5, labels))
    calinski.append(calinski_harabasz_score(X_pca5, labels))

results = pd.DataFrame({
    'K': list(K_range),
    'WCSS (elbow)': np.round(wcss, 1),
    'Silhouette': np.round(silhouette, 3),
    'Davies-Bouldin': np.round(davies, 3),
    'Calinski-Harabasz': np.round(calinski, 1),
})
print(results.to_string(index=False))


In [ ]:
# Plot all four so you can compare what each one 'votes' for.
fig = make_subplots(rows=2, cols=2, subplot_titles=(
    'Elbow / WCSS (look for the bend)',
    'Silhouette (higher = better)',
    'Davies-Bouldin (lower = better)',
    'Calinski-Harabasz (higher = better)'))

fig.add_scatter(x=list(K_range), y=wcss, mode='lines+markers', row=1, col=1)
fig.add_scatter(x=list(K_range), y=silhouette, mode='lines+markers', row=1, col=2)
fig.add_scatter(x=list(K_range), y=davies, mode='lines+markers', row=2, col=1)
fig.add_scatter(x=list(K_range), y=calinski, mode='lines+markers', row=2, col=2)

fig.update_layout(height=650, showlegend=False,
                  title_text='Four ways to choose K — do they agree?')
fig.show()


**How to read this:**
- **Elbow (WCSS)**: it always decreases as K grows (more clusters always fit tighter). You look for the "elbow" — the bend where adding more clusters stops helping much. Subjective by design. This is the oldest and fuzziest method — *not wrong, just imprecise*.
- **Silhouette**: measures how well-separated clusters are. Higher is better. More rigorous than elbow. Widely used.
- **Davies-Bouldin**: ratio of within-cluster scatter to between-cluster separation. *Lower* is better.
- **Calinski-Harabasz**: variance ratio. *Higher* is better.

**The real lesson:** when several independent metrics point at the same K, you can be reasonably confident. When they disagree, there's no single truth — you pick based on what's *useful* for the business.

🏭 Industry note: in practice the number of clusters is very often a **business decision**. If the NGO can realistically run 4 distinct aid programs, K=4 is the answer regardless of what the math whispers. The metrics inform the choice; the use-case usually decides it. This is the "no single best" idea again — judgment over formula.


## §7 Final model — fit K-Means with the chosen K

We'll go with **K=4** (well-supported by the metrics and a sensible number of aid tiers).


In [ ]:
optimal_k = 4

kmeans = KMeans(n_clusters=optimal_k, init='k-means++',
                n_init=10, random_state=RANDOM_STATE)
cluster_labels = kmeans.fit_predict(X_pca5)

# Attach cluster labels back to the ORIGINAL (un-transformed) data,
# so we can interpret clusters using real-world units (dollars, years, etc.).
df_clustered = df.copy()
df_clustered['Cluster'] = cluster_labels

print("Countries per cluster:")
print(df_clustered['Cluster'].value_counts().sort_index())


In [ ]:
# Visualize the clusters in the first two principal components.
viz = pd.DataFrame({
    'PC1': X_pca5[:, 0],
    'PC2': X_pca5[:, 1],
    'Cluster': cluster_labels.astype(str),  # string -> plotly treats as categories
    'Country': country_names,
})
fig = px.scatter(viz, x='PC1', y='PC2', color='Cluster',
                 hover_data=['Country'],
                 title=f'Clusters in PC1-PC2 space (K={optimal_k})',
                 color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_traces(marker=dict(size=10, line=dict(width=1, color='white')))
fig.update_layout(height=550)
fig.show()


## §8 Interpret — turn clusters into meaning

A cluster labeled "0" means nothing to the NGO. We have to *describe* each cluster in plain terms: what kind of country falls into each group? We do this by averaging the original features within each cluster.


In [ ]:
# Average each real-world feature per cluster.
cluster_profile = df_clustered.groupby('Cluster').mean(numeric_only=True).round(2)
print("Average feature values per cluster:")
print(cluster_profile)


In [ ]:
# A heatmap makes the profile readable at a glance.
# Normalize each feature 0-1 across clusters so colors are comparable.
profile_norm = (cluster_profile - cluster_profile.min()) / \
               (cluster_profile.max() - cluster_profile.min())

fig = px.imshow(profile_norm.T, text_auto='.2f', aspect='auto',
                color_continuous_scale='RdYlGn',
                title='Cluster profiles (green = high, red = low, per feature)',
                labels=dict(x='Cluster', y='Feature', color='Relative level'))
fig.update_layout(height=500)
fig.show()


**Reading the heatmap to name the clusters:** find the cluster that is RED on the good things (income, life_expec, gdpp, health) and GREEN/high on the bad things (child_mort, total_fer). *That* cluster is the "most in need" group — the NGO's target.

🏭 Industry note: this naming step is where you'd loop in a *domain expert* (here, someone from the NGO). The model finds groups; humans decide what they mean and what to do. That human-in-the-loop step is real industry practice, not a shortcut.


In [ ]:
# Identify the 'most in need' cluster automatically: the one with the
# HIGHEST average child mortality (a strong proxy for being underdeveloped).
neediest_cluster = cluster_profile['child_mort'].idxmax()
print(f"The 'most in need' cluster is: Cluster {neediest_cluster}")
print(f"  Avg child mortality: {cluster_profile.loc[neediest_cluster, 'child_mort']}")
print(f"  Avg income:          {cluster_profile.loc[neediest_cluster, 'income']}")
print(f"  Avg life expectancy: {cluster_profile.loc[neediest_cluster, 'life_expec']}")


In [ ]:
# World map: color each country by its cluster. Instantly shows the
# geography of need (you'll see the neediest cluster concentrate in regions).
fig = px.choropleth(
    df_clustered, locations='country', locationmode='country names',
    color=df_clustered['Cluster'].astype(str), hover_name='country',
    hover_data=['child_mort', 'income', 'gdpp', 'life_expec'],
    title='Country clusters on the world map',
    color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(height=600)
fig.show()


## §9 Deliver — the ranked priority list

Clustering put countries into tiers. But the NGO wants a *ranked* list within the neediest tier — who gets help first. We build a single "need score" from the most relevant indicators.


In [ ]:
# Focus on the neediest cluster only.
needy = df_clustered[df_clustered['Cluster'] == neediest_cluster].copy()

# Build a composite need score. We standardize within this group, then
# combine: bad indicators add to need (+), good indicators subtract (-).
score_features = ['child_mort', 'income', 'life_expec', 'gdpp', 'total_fer']
needy_scaled = StandardScaler().fit_transform(needy[score_features])

needy['need_score'] = (
    needy_scaled[:, 0]   # child_mort  (higher = worse  -> +)
    - needy_scaled[:, 1] # income      (higher = better -> -)
    - needy_scaled[:, 2] # life_expec  (higher = better -> -)
    - needy_scaled[:, 3] # gdpp        (higher = better -> -)
    + needy_scaled[:, 4] # total_fer   (higher = worse  -> +)
)

top_priority = needy.sort_values('need_score', ascending=False).head(15)
print("Top 15 countries most in need of aid:")
print(top_priority[['country', 'child_mort', 'income',
                    'gdpp', 'life_expec', 'need_score']].to_string(index=False))


**That's the deliverable.** A defensible, ranked list the NGO can act on — built from raw data, through cleaning, EDA, preprocessing, modeling, and interpretation. The whole pipeline, end to end.

🏭 Industry note: the hand-built `need_score` with manual weights is fine for a first pass, but in a real project you'd justify those weights with the domain expert (why does child_mort count as much as income?). Arbitrary weights are a common place for stakeholders to push back.


## §10 🏭 What would be different in a real production setting?

You did a complete *analysis*. A production *system* adds a few layers on top. You don't need these now, but knowing they exist completes your mental map:

| Layer | What it adds | Tools you'd hear about |
|---|---|---|
| **Reproducibility** | Anyone can re-run and get your exact result | `random_state`, pinned package versions, `requirements.txt` |
| **Pipelines** | Preprocessing + model as one saveable object | `sklearn.Pipeline` (shown in §4) |
| **Experiment tracking** | Log every run's params and scores | MLflow, Weights & Biases |
| **Data versioning** | Know exactly which data produced which result | DVC, lakeFS |
| **Serving** | Expose the model so other systems can use it | FastAPI, Docker |
| **Monitoring** | Alert when new data drifts from training data | Evidently, custom dashboards |

The analysis (what you did) is the hard, valuable, *thinking* part. The production layers are mostly engineering plumbing you learn on the job. Don't let the long list intimidate you — each item is small once you meet it.


## §11 Your three questions, answered in one place

**Q: Is PCA necessary?**
No — not for this project (§5 proved it: clustering with vs without PCA agreed almost perfectly). PCA earns its place with many features, heavy redundancy, or a need to visualize. Here it's a *visualization choice*, not a modeling requirement. "Always do PCA first" is a myth — it can even hurt, since it optimizes variance, not cluster separation.

**Q: Are elbow & silhouette the best, or outdated?**
Neither outdated nor uniquely "best." Clustering has no ground truth, so all cluster-count metrics are heuristics. We looked at four (elbow, silhouette, Davies-Bouldin, Calinski-Harabasz). When they agree, trust it; when they disagree, the business use-case decides. The number of clusters is frequently a business call, not a math one.

**Q: (the mindset) "What is the *best* method?" — coming from CS**
This is the most important one. Software engineering usually has a best tool. ML/statistics usually doesn't — it's tradeoffs and fit-for-purpose. Searching for "the one best method" will keep you feeling behind because it doesn't exist. The senior skill is *judgment about tradeoffs given your data, goal, and constraints* — which you already showed when you questioned your own feature choices. That instinct IS the skill.


## §12 Your next step — from grouping to predicting

Everything above is **unsupervised** (discovering structure). The other half of ML is **supervised** (predicting a target). You're ready for it, and you can use this exact dataset.

**Challenge for your next session:**
> Predict `life_expec` (life expectancy) from the other numeric features using **Linear Regression**.

You'll reuse ~80% of this notebook (load, inspect, EDA, scale). The new 20%:
1. Split data into train and test sets (`train_test_split`)
2. Fit `LinearRegression().fit(X_train, y_train)`
3. Predict on the test set
4. Evaluate with R² and Mean Absolute Error
5. Plot predicted vs actual

This is where the **Normal distribution** finally connects to a model: linear regression *assumes the prediction errors are Normal*. You'll have heard that music, and now you'll play it.

When you're ready, bring this notebook and we'll build the supervised version together — and you'll have covered both halves of classical ML on a dataset you fully understand.
